# ConvNeXt Biomass Pipeline

This notebook fine-tunes pretrained ConvNeXt models on the biomass images, evaluates them with leakage-aware cross-validation, and produces predictions for the single test image.

We train the network on the three independent targets (`Dry_Clover_g`, `Dry_Dead_g`, and `Dry_Green_g`) and then reconstruct the two dependent targets (`GDM_g` and `Dry_Total_g`). That keeps the final five predictions internally consistent instead of letting the model predict biologically impossible combinations.

## 1. Import PyTorch, torchvision models, and the reusable project pipeline

The first code cell keeps setup explicit. We import the standard analysis libraries, add `src` to the Python path so notebook execution works from different working directories, and then import the reusable ConvNeXt helpers from the project module.

A small naming note: ConvNeXt lives in `torchvision.models`, which is the image-model part of the PyTorch ecosystem.

In [9]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from torchvision import models


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for base in (start, *start.parents):
        if (base / "src").exists():
            return base
    raise FileNotFoundError("Could not find the repository root containing the src directory.")


repo_root = find_repo_root()
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from main.vision.ConvNeXt.convnext import (
    ALL_TARGET_COLUMNS,
    BASE_TARGET_COLUMNS,
    MODEL_REGISTRY,
    ConvNeXtTrainingConfig,
    infer_device,
    load_test_frame,
    load_training_frame,
    resolve_data_root,
    run_convnext_experiment,
    save_experiment_outputs,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

print(f"torch version: {torch.__version__}")
print(f"available ConvNeXt models: {list(MODEL_REGISTRY)}")
print(f"device selected by PyTorch: {infer_device().type}")


torch version: 2.10.0+cpu
available ConvNeXt models: ['convnext_tiny', 'convnext_small', 'convnext_base', 'convnext_large']
device selected by PyTorch: cpu


## 2. Load one row per image and inspect the leakage boundary

The project stores training labels in long format, with one row per `(image, target_name)` pair. The next cell reshapes that into one row per image so the training pipeline sees a single image with its target vector.

We also inspect `collection_group`, which is the grouping key used during cross-validation. Grouping by sampling date, state, and species keeps visually or biologically similar collection batches from leaking across train and validation folds.

In [10]:
data_root = resolve_data_root(repo_root)
train_frame = load_training_frame(data_root)
test_frame = load_test_frame(data_root)

# These checks document why we predict three targets directly and rebuild two of them.
gdm_gap = (train_frame["GDM_g"] - (train_frame["Dry_Green_g"] + train_frame["Dry_Clover_g"])).abs().max()
dry_total_gap = (
    train_frame["Dry_Total_g"]
    - (train_frame["Dry_Dead_g"] + train_frame["Dry_Green_g"] + train_frame["Dry_Clover_g"])
).abs().max()

print(f"data root: {data_root}")
print(f"train images: {len(train_frame)}")
print(f"test images: {len(test_frame)}")
print(f"unique collection groups: {train_frame['collection_group'].nunique()}")
print(f"max GDM reconstruction gap in labels: {gdm_gap:.6f}")
print(f"max Dry_Total reconstruction gap in labels: {dry_total_gap:.6f}")

display(train_frame.head())
display(train_frame["collection_group"].value_counts().head(10).rename("images_in_group").to_frame())


data root: C:\.repos\inf367a_image2biomass\src\data
train images: 357
test images: 1
unique collection groups: 40
max GDM reconstruction gap in labels: 0.000100
max Dry_Total reconstruction gap in labels: 0.308800


,image_path,Sampling_Date,State,Species,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g,image_relpath,collection_group
0,C:\.repos\inf367a_image2biomass\src\data\train...,2015/9/4,Tas,Ryegrass_Clover,0.0000,31.9984,16.2751,48.2735,16.2750,train/ID1011485656.jpg,2015/9/4 | Tas | Ryegrass_Clover
1,C:\.repos\inf367a_image2biomass\src\data\train...,2015/4/1,NSW,Lucerne,0.0000,0.0000,7.6000,7.6000,7.6000,train/ID1012260530.jpg,2015/4/1 | NSW | Lucerne
2,C:\.repos\inf367a_image2biomass\src\data\train...,2015/9/1,WA,SubcloverDalkeith,6.0500,0.0000,0.0000,6.0500,6.0500,train/ID1025234388.jpg,2015/9/1 | WA | SubcloverDalkeith
3,C:\.repos\inf367a_image2biomass\src\data\train...,2015/5/18,Tas,Ryegrass,0.0000,30.9703,24.2376,55.2079,24.2376,train/ID1028611175.jpg,2015/5/18 | Tas | Ryegrass
4,C:\.repos\inf367a_image2biomass\src\data\train...,2015/9/11,Tas,Ryegrass,0.4343,23.2239,10.5261,34.1844,10.9605,train/ID1035947949.jpg,2015/9/11 | Tas | Ryegrass


,images_in_group
collection_group,
2015/5/18 | Tas | Ryegrass,22
2015/6/26 | Tas | Ryegrass_Clover,18
2015/11/10 | Tas | Ryegrass_Clover,17
2015/7/8 | WA | Clover,12
2015/9/29 | Vic | Ryegrass_Clover,11
2015/11/9 | Tas | Clover,11
2015/8/14 | Vic | Ryegrass_Clover,11
2015/10/13 | Vic | Phalaris_BarleyGrass_SilverGrass_SpearGrass_Clover_Capeweed,11
2015/10/6 | NSW | Fescue,11


## 3. Choose the training configuration

This cell keeps the experiment settings in one place. We evaluate every torchvision ConvNeXt size, fine-tune pretrained weights, and use grouped cross-validation to avoid leakage.

The default epochs and batch size are intentionally conservative on CPU so the notebook remains runnable in this environment. If you move to a GPU, increasing them is the easiest next step.

In [11]:
device = infer_device().type

# CPU-friendly defaults keep the notebook practical while we build the pipeline.
epochs = 2 if device == "cpu" else 12
batch_size = 4 if device == "cpu" else 16
patience = 1 if device == "cpu" else 3

config = ConvNeXtTrainingConfig(
    data_root=data_root,
    model_names=tuple(MODEL_REGISTRY.keys()),
    n_splits=5,
    epochs=epochs,
    batch_size=batch_size,
    patience=patience,
    use_pretrained=True,
    device=device,
)

config_summary = pd.Series(
    {
        "device": config.device,
        "models": ", ".join(config.model_names),
        "independent_targets": ", ".join(BASE_TARGET_COLUMNS),
        "all_reported_targets": ", ".join(ALL_TARGET_COLUMNS),
        "n_splits": config.n_splits,
        "epochs": config.epochs,
        "batch_size": config.batch_size,
        "patience": config.patience,
        "pretrained": config.use_pretrained,
        "output_dir": str(config.output_dir),
    }
)

display(config_summary.to_frame(name="value"))


,value
device,cpu
models,"convnext_tiny, convnext_small, convnext_base, ..."
independent_targets,"Dry_Clover_g, Dry_Dead_g, Dry_Green_g"
all_reported_targets,"Dry_Clover_g, Dry_Dead_g, Dry_Green_g, GDM_g, ..."
n_splits,5
epochs,2
batch_size,4
patience,1
pretrained,True
output_dir,C:\.repos\inf367a_image2biomass\src\main\model...


## 4. Run leakage-aware grouped cross-validation and fine-tune every ConvNeXt size

The experiment function does the heavy lifting:

- It splits folds with `GroupKFold` on `collection_group`.
- It fits the target scaler on the training fold only, which prevents label leakage.
- It fine-tunes pretrained ConvNeXt weights end to end by replacing the classifier head and updating the full network.
- It stores out-of-fold predictions for honest validation metrics and averages fold-level test predictions for the held-out test image.

This is the slowest cell in the notebook.

In [ ]:
summary_frame, results = run_convnext_experiment(config)
save_experiment_outputs(summary_frame=summary_frame, results=results, output_dir=config.output_dir)

print(f"Saved experiment artifacts to: {config.output_dir}")


## 5. Compare model sizes with cross-validation metrics

The summary table lets us compare the ConvNeXt variants on the same grouped folds. The most important headline metric here is `mean_weighted_r2`, because it mirrors the competition's target weighting more closely than a plain average across targets.

In [ ]:
summary_frame


## 6. Inspect the predicted target values for the test image

Each model produces a full five-target prediction for the one test image. We collect those predictions side by side, then identify the best model according to cross-validation and preview the submission rows it would write.

In [ ]:
test_predictions_by_model = pd.concat(
    [
        artifacts.test_predictions.assign(model_name=model_name)
        for model_name, artifacts in results.items()
    ],
    ignore_index=True,
)

ordered_columns = ["model_name", "image_relpath", *ALL_TARGET_COLUMNS]
test_predictions_by_model = test_predictions_by_model[ordered_columns]

best_model_name = summary_frame.iloc[0]["model_name"]
best_submission = results[best_model_name].submission

print(f"best model by grouped CV: {best_model_name}")
display(test_predictions_by_model)
display(best_submission)


## 7. Inspect fold-by-fold diagnostics for a chosen model

The last cell helps us debug training stability. If a model's mean score looks good but one fold collapses, that usually means the model is sensitive to certain collection groups, which is exactly the kind of leakage-resistant behavior we want to understand.

In [ ]:
results[best_model_name].fold_metrics.sort_values("fold")
